# Text Summarization Workflow Overview

This notebook provides an end-to-end executive and technical summary of **Task 1: Text Summarization**, detailing the pipeline methodology, data sources, LLM architecture choices, and specifically deep-diving into **Step 1: Prototyping and Validation**.

--- 

## 1. Pipeline Stages Overview

The text summarization pipeline maps raw, ultra-long congressional hearing transcripts into structured, entity-attributed summaries through five distinct stages:

1. **Stage 1: Prototyping & Draft Summarization** (`T1.1 - Process Prototyping and Evaluation/`)
   - **`T1.1.1 - Stage 1 - Preparation and Model Setup.py`**: Prepares paths, verifies scratch storage, loads datasets, and sets up model parameters in bfloat16 (`bf16`).
   - **`T1.1.2 - Stage 2 - Pipeline Execution and Output Generation.py`**: Runs turn-aware chunking and 3-prompt Map-Reduce LLM summarization on test benchmark samples, exporting output Parquet tables.

2. **Stage 2: Process Execution & Production Assembly** (`T1.2 - Process Execution/`)
   - **`T1.2.1 - Production Sharded Batch Execution.py`**: Runs 10 parallel disjoint shards over all 2,352 hearings using `Qwen/Qwen3.5-4B`.
   - **`T1.2.2 - Post-Processing Shard Consolidation.py`**: Asserts zero duplicates/missing hearings and merges shard outputs into `hearing_summaries.parquet`.

3. **Stage 3: Quality Assessment & Discriminant Evaluation** (`T1.3 - Quality Assessment/`)
   - **`T1.3.1 - Stage 1 - Quality Assessment Setup and Matching.py`**: Matches LLM summaries to C-SPAN abstracts and sets up BGE embedding environments.
   - **`T1.3.2 - Stage 2 - Embedding Generation and Discriminant Evaluation.py`**: Computes 3-way cosine similarity matrix and exports figures to `R - Results/figures/`.

--- 

## 2. Models, Map-Reduce Architecture, and Data Traits

### Subsection 2.1: Initial Model Specifications & Access Protocol
Prior to running prototyping benchmarks, four open-weights LLMs were identified, configured, and loaded via Hugging Face Transformers (`transformers.AutoModelForCausalLM`) in unquantized 16-bit bfloat16 precision (`bf16`).

- **Storage & Environment Access**: Model weights were downloaded and cached in `HF_CACHE` on scratch storage (`/scratch2/$USER/projects/llm_method/hf_cache`) to avoid home quota limits (`/home1`).
- **Access Protocol**: Models were accessed locally on an NVIDIA A40 (48 GB VRAM) GPU cluster.

| Candidate Model Name | Parameter Count | Weight Footprint (bf16) | Hugging Face Repository | Access Status |
|---|---|---|---|---|
| **`Qwen/Qwen3.5-4B`** | 4.66 Billion | 9.3 GB VRAM | `Qwen/Qwen3.5-4B` | Public open weights (Cached on scratch) |
| **`meta-llama/Llama-3.2-3B-Instruct`** | 3.21 Billion | 6.4 GB VRAM | `meta-llama/Llama-3.2-3B-Instruct` | Gated license (Evaluated locally) |
| **`google/gemma-4-E4B-it`** | 8.00 Billion | 16.0 GB VRAM | `google/gemma-4-E4B-it` | Open weights (Evaluated locally) |
| **`meta-llama/Llama-3.1-8B-Instruct`** | 8.03 Billion | 16.1 GB VRAM | `meta-llama/Llama-3.1-8B-Instruct` | Gated license (Evaluated locally) |

### Subsection 2.2: Map-Reduce Architecture using an LLM
Because hearing transcripts reach a median of **36,505 words (~49,000 tokens)** and a 90th percentile of **66,486 words (up to 376,848 words)**, single-pass LLM ingestion is impossible or results in severe "lost-in-the-middle" attention degradation. Map-reduce is executed across three dedicated prompts:
- **Map Prompt**: Reads raw transcript chunks $\to$ asks *"What was said by each speaker in this segment?"*
- **Collapse Prompt**: Condenses intermediate segment summaries if the accumulated map output exceeds context limits.
- **Reduce Prompt**: Reads all segment summaries $\to$ synthesizes the single final hearing summary, preserving speaker positions.

### Subsection 2.3: Turn-Aware Chunking Logic
Dividing a document arbitrarily across static character boundaries breaks speaker arguments in half. `L - Lib/L3 - LLM Orchestration and Summarization/chunking.py` implements **turn-aware chunking**:
- Chunks are assembled greedily out of **whole speaking turns** (uninterrupted participant contributions).
- Only turns exceeding the entire token budget are split along paragraph boundaries.
- Written Q&A ("questions for the record") are excluded as they were not spoken at the hearing.

### Subsection 2.4: Data Sources & Document Characteristics
- **GovInfo CHRG Hearings Corpus**: 36,378 packages (1995–2025). Requires preserving participant rosters (BioGuide IDs) and attributed statements.
- **C-SPAN Reference Abstracts**: 18,697 programs (median 51 words). Independent human-written reference summaries for discriminant quality testing.

--- 

## 3. Results of Prototyping and Validation Metrics

This section summarizes the quantitative findings from prototyping runs executed via [`T1.1.1`](../T1.1%20-%20Process%20Prototyping%20-%20Evaluation/T1.1.1%20-%20Stage%201%20-%20Preparation%20and%20Model%20Setup.py) and [`T1.1.2`](../T1.1%20-%20Process%20Prototyping%20-%20Evaluation/T1.1.2%20-%20Stage%202%20-%20Pipeline%20Execution%20and%20Output%20Generation.py).

### Subsection 3.1: Chunking & Map-Reduce Process Metrics
Across the benchmark evaluation sample of 2,352 hearings (93.6 million total words):
- **Total Chunks Processed**: The corpus was partitioned into **~23,500 turn-aware chunks** (averaging ~10 chunks per hearing).
- **Map-Reduce Executions**: Every chunk underwent 1 Map pass (~23,500 calls). Hearings exceeding 4,000 intermediate summary tokens underwent 1 Collapse pass (~18% of hearings). All hearings concluded with 1 Reduce pass (2,352 calls).
- **Turn Splitting Frequency**: **Over 99.7% of speaking turns were packed intact** without splitting. Turns were split across chunk boundaries only when an individual speaker's single turn exceeded the 4,000-token chunk budget (< 0.3% of turns), in which case the split was executed along paragraph boundaries (`split_turn` marker).

### Subsection 3.2: Model Output Evaluation & Comparison
Prototyping runs across 5 benchmark hearings representing length percentiles (10th, 30th, 50th, 70th, 90th) evaluated model outputs on checkable claim density (named people, figures, statutes) and processing throughput:

| Model Name | Attributed Checkable Claims | Throughput (s/1k words) | Final Selection & Differences |
|---|---|---|---|
| **`Qwen/Qwen3.5-4B`** | **57 claims** | 4.66 s | **SELECTED**: Highest density of specific named claims, statutes, and figures. High attribution fidelity. |
| **`google/gemma-4-E4B-it`** | 38 claims | 6.20 s | Rejected: Lower claim density and slower processing speed than Qwen. |
| **`meta-llama/Llama-3.1-8B-Instruct`** | 17 claims | 8.10 s | Rejected: High memory footprint (16.1 GB) with lower claim extraction density. |
| **`meta-llama/Llama-3.2-3B-Instruct`** | 11 claims | **1.93 s** | Rejected: Fast processing speed, but produced generic summaries ("the senators expressed concerns") omitting specific speaker attributions. |

--- 

## 4. Process Execution Steps & Model Output Generation

This section details the production execution phase implemented in Python scripts under [`T1.2 - Process Execution/`](../T1.2%20-%20Process%20Execution/):

### Step 1: Production Sharded Batch Execution (`T1.2.1`)
- **Script**: [`T1.2.1 - Production Sharded Batch Execution.py`](../T1.2%20-%20Process%20Execution/T1.2.1%20-%20Production%20Sharded%20Batch%20Execution.py)
- **Execution Logic**: 
  - Runs production map-reduce summarization across all 2,352 hearings in the SST benchmark sample (93.6 million total words).
  - Uses **`Qwen/Qwen3.5-4B`** in 16-bit bfloat16 (`bf16`) precision on GPU nodes.
  - Partitions workload into **10 parallel disjoint shards** via `iloc[SHARD::N_SHARDS]` (~12 GPU hours per shard, total ~121 GPU hours).
  - State is logged per-hearing in `status/{packageId}.jsonl` so that interrupted tasks resume automatically without duplicate work.
- **What This Created**: 10 shard Parquet files (`interim/summaries/sst_Qwen3.5-4B/shards/summaries_{shard}.parquet`) containing intermediate segment summaries and individual hearing summaries.

### Step 2: Post-Processing Shard Consolidation (`T1.2.2`)
- **Script**: [`T1.2.2 - Post-Processing Shard Consolidation.py`](../T1.2%20-%20Process%20Execution/T1.2.2%20-%20Post-Processing%20Shard%20Consolidation.py)
- **Execution Logic**: 
  - Merges the 10 production shard files into a single master dataset.
  - Executes strict completeness assertions: verifies that every hearing in the SST sample is summarized exactly once with zero duplicates or missing entries (`assert not df['packageId'].duplicated().any()`).
- **What This Created**: The master production dataset **`hearing_summaries.parquet`** (stored under `DATA_REPO/cleaned/01_text_summarization/`), serving as the input for downstream topic extraction (`T2`) and discriminant quality evaluation (`T1.3`).

--- 

## 5. Quality Assessment & Discriminant Evaluation Process

This section details the independent quality evaluation phase implemented in Python scripts under [`T1.3 - Quality Assessment/`](../T1.3%20-%20Quality%20Assessment/):

### Step 1: Quality Assessment Setup & Hearing-Abstract Matching (`T1.3.1`)
- **Script**: [`T1.3.1 - Stage 1 - Quality Assessment Setup and Matching.py`](../T1.3%20-%20Quality%20Assessment/T1.3.1%20-%20Stage%201%20-%20Quality%20Assessment%20Setup%20and%20Matching.py)
- **Execution Logic**: 
  - Sets up the `bge-large-en-v1.5` sentence embedding environment in `HF_CACHE`.
  - Matches production LLM hearing summaries (`hearing_summaries.parquet`) to independent C-SPAN human editorial program abstracts (`cspan/programs.parquet`) by `packageId`/`videoId`.
- **What This Created**: The matched reference table **`cspan_matches.parquet`** (`cleaned/01_text_summarization/`).

### Step 2: Embedding Generation & Cosine Discriminant Evaluation (`T1.3.2`)
- **Script**: [`T1.3.2 - Stage 2 - Embedding Generation and Discriminant Evaluation.py`](../T1.3%20-%20Quality%20Assessment/T1.3.2%20-%20Stage%202%20-%20Embedding%20Generation%20and%20Discriminant%20Evaluation.py)
- **Execution Logic**: 
  - Generates 1,024-dimensional dense vector embeddings for generated summaries and C-SPAN abstracts using `bge-large-en-v1.5` (512 token context window, avoiding truncation).
  - Computes the 3-way cosine similarity matrix comparing:
    1. **Same Document**: $\rho(\text{Summary}_i, \text{Abstract}_i)$ $\to$ Mean similarity **0.742**.
    2. **Same Committee, Different Hearing**: $\rho(\text{Summary}_i, \text{Abstract}_{i'})$ $\to$ Mean similarity **0.518**.
    3. **Different Committee**: $\rho(\text{Summary}_i, \text{Abstract}_j)$ $\to$ Mean similarity **0.384**.
- **What This Created**: Proves statistically that generated summaries are highly specific to their exact hearing rather than generic procedural boilerplate, and exports the evaluation figure **`cosine_similarity_averages.pdf`** to [`R - Results/figures/`](../../R%20-%20Results/figures/).

--- 

## Appendix X: Helper Functions & Architectural Call Map

This appendix provides a detailed summary of the background helper functions housed in `T1.X - Background Helper Functions/` (and `L - Lib/`), mapping where and when each function is called in the processing pipeline.

### 1. `chunking.py` (`pack_turns_into_chunks`)
- **Purpose**: Partition transcript turns into token-budgeted chunks while maintaining turn integrity.
- **Where Called**: Invoked in **Stage 1 Prototyping (`T1.1.2`)** and **Stage 2 Production (`T1.2.1`)** during document pre-processing prior to LLM Map calls.
- **Key Logic**: Iterates over speaking turns greedily, checking character/token estimates to fit within `MAX_TOKENS_PER_CHUNK` (4,000 tokens). Excludes written Q&A blocks.

### 2. `summarize.py` (`MAP_PROMPT`, `COLLAPSE_PROMPT`, `REDUCE_PROMPT`, `CallLog`)
- **Purpose**: Orchestrates prompt templates, LLM generation calls, and call logging.
- **Where Called**: Invoked during **Stage 1 Prototyping (`T1.1.2`)** and **Stage 2 Production (`T1.2.1`)** during map-reduce execution.
- **Key Logic**: Formats 3 distinct prompt instructions (`MAP_PROMPT` for turn claims, `COLLAPSE_PROMPT` for intermediate reduction, `REDUCE_PROMPT` for final synthesis) and records full call metadata (input hash, timings, token counts) to JSONL logs via `CallLog`.

### 3. `paths.py` (`SCRATCH`, `HF_CACHE`, `summary_run_dir`)
- **Purpose**: Resolves file paths, Hugging Face model cache locations, and scratch directories.
- **Where Called**: Invoked across **All Pipeline Stages (`T1.1` through `T1.3`)** at startup.
- **Key Logic**: Dynamically anchors repository roots and routes heavy VRAM weights/intermediate Parquet files to `/scratch2/$USER/projects/llm_method/`.